### Gold Layer

In [1]:
df = spark.sql("SELECT * FROM LH_Discharge.Silver.silvertable")
display(df)

StatementMeta(, 2e706878-2cca-479a-8f4d-cb70b02726a7, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9480a0e1-59cb-41a0-be25-81d6a0da5079)

In [2]:
# create single column dimension table with hospital name

dim_hospital = df.select('Facility_Name').distinct().withColumnRenamed('Facility_Name','Hospital_Name')
dim_hospital.write.format('delta').saveAsTable('LH_Discharge.Gold.dim_hospital')

StatementMeta(, 2e706878-2cca-479a-8f4d-cb70b02726a7, 4, Finished, Available, Finished, False)

In [8]:
# summary table to speed up the process in gold layer
from pyspark.sql.functions import avg, round

df_summ = df.groupBy('Facility_Name','Age_Group').\
           agg(round(avg('Total_Charges'),2).alias('Avg_Cost_Per_Stay'),
           round(avg('Length_of_Stay'),2).alias('Avg_Stay'))

display(df_summ)



StatementMeta(, 2e706878-2cca-479a-8f4d-cb70b02726a7, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 303c08f6-17fb-47e9-873e-f4c85250c224)

In [9]:
df_summ.write.format('delta').saveAsTable('LH_Discharge.Gold.gold_summary')

StatementMeta(, 2e706878-2cca-479a-8f4d-cb70b02726a7, 11, Finished, Available, Finished, False)

In [11]:
# Adding profit_margin and high cost flag in silver dataframe
from pyspark.sql.functions import col, when
df_enrich = df.\
            withColumn('Profit_Margin', col('Total_Charges')-col('Total_Costs')).\
            withColumn('High_Cost_Flag', when(col('Total_Charges') > 50000, 'Y').otherwise('N'))

display(df_enrich)

StatementMeta(, 2e706878-2cca-479a-8f4d-cb70b02726a7, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b03d90b5-e4aa-4b34-b591-f1313bceecd9)

In [12]:
df_enrich.write.format('delta').saveAsTable('LH_Discharge.Gold.gold_table')

StatementMeta(, 2e706878-2cca-479a-8f4d-cb70b02726a7, 14, Finished, Available, Finished, False)